# 06 · The four predictor sets

The study's first question is how many predictors a model needs. Every model is fitted on four
sets, from the largest to the smallest, and compared on the same held-out women:

| Set | What it is |
|---|---|
| **Full pool** | the clean pool of 86 (notebook 05) |
| **21-predictor set** | the 21 predictors built in notebook 01 |
| **Short set** | the predictors that clear a strict bar in repeated forward selection |
| **Parsimonious set** | a short list chosen for a web form, by forward selection with the helper model |

**The short set.** Starting from nothing, the predictor that most raises the out-of-fold AP of a
lightly regularised logistic regression is added, over five different fold seeds. A step is
kept only if it adds at least 0.005 AP on average **and** helps on at least four of the five
seeds. Tier 1 is the steps until 90% of the full pool's gain over chance is reached; Tier 2 the
remaining kept steps. The short set is Tier 1 plus Tier 2. Holding the model fixed and simple
is deliberate: the question is which variables are worth collecting, not which model is best.

**The parsimonious set.** Greedy forward selection with the helper LightGBM of notebook 04 (at
most 300 trees), on out-of-fold AP. The ANC8+ search was the study's first: all 95 candidates
at every step, up to ten steps. It showed how much such a search overfits (its own development
AP ran far above what the list scored on held-out women), so the searches for ANC1+ and ANC4+
first pre-screen to the 30 strongest candidates and stop at eight predictors. Both versions are
reproduced exactly as they ran. A predictor the search chose that notebook 05 found to be a
duplicate is replaced by the copy the clean pool keeps.

**Time.** The short set takes about 30 minutes per rung. The parsimonious search takes about
four hours for ANC8+ (every candidate at every step) and one to two hours each for ANC1+ and
ANC4+. Set `RESELECT = False` to take the study's sets and skip both searches.

**Writes:** `outputs/feature_value/full/<rung>/summary.json`, `outputs/enhanced/<rung>/parsimonious_spec.json`

In [ ]:
# Settings for this notebook
QUICK = False        # True: a smoke run with 3 search trials per model; numbers will not match the study
FOLLOW_STUDY = True  # when a re-derived choice differs from the study's, carry the study's forward

import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd()
sys.path.insert(0, str(here.parent / "src" if here.name.startswith("notebooks") else here / "src"))
from anc8 import repro
P = repro.setup(quick=QUICK)
QUICK = P.quick

import numpy as np
import pandas as pd
from anc8 import library as lib, tuning

from anc8 import parsimony
RESELECT = not QUICK   # False: take the study's sets and skip both searches
study = repro.reference("predictor_sets.json")

## The short set, per rung

In [ ]:
for rung in lib.RUNGS:
    out = P.outputs / "feature_value" / "full" / rung
    if RESELECT:
        repro.run_script("run_feature_value.py", "--cohort", "full", "--outcome", rung, "--stages", "2", "--bootstrap", "0")
        s = json.load(open(out / "summary.json"))
        got = {"tier1": s["tier1"], "tier2": s["tier2"]}
        same = got == study["short_tiers"][rung]
        repro.check(f"{rung}: the short set is the study's", same, "" if same else str(got))
        if FOLLOW_STUDY and not same:
            s.update(study["short_tiers"][rung]); json.dump(s, open(out / "summary.json", "w"), indent=2)
    else:
        out.mkdir(parents=True, exist_ok=True)
        json.dump(study["short_tiers"][rung], open(out / "summary.json", "w"), indent=2)
    s = json.load(open(out / "summary.json"))
    print(f"{rung}: tier 1 {s['tier1']}, tier 2 {s['tier2']}")

## The parsimonious set, per rung

In [ ]:
for rung in lib.RUNGS:
    out = P.outputs / "enhanced" / rung
    if RESELECT:
        d = lib.data("pool95", rung)
        best = json.load(open(out / "best_hyperparameters.json"))["LightGBM"]
        prescreen, steps = parsimony.SETTINGS[rung]
        chosen, path = parsimony.forward(d.X_dev, d.y_dev, d.w_dev, d.splits, d.numeric, d.categorical,
                                         parsimony.selector_params(best), prescreen=prescreen, max_steps=steps)
        same = chosen == study["pars_raw"][rung]
        repro.check(f"{rung}: the parsimonious search picks the study's list", same, "" if same else str(chosen))
        if FOLLOW_STUDY and not same:
            chosen = study["pars_raw"][rung]
    else:
        chosen = study["pars_raw"][rung]
    json.dump({"predictors": chosen, "base_model": "LightGBM"}, open(out / "parsimonious_spec.json", "w"), indent=2)

## The four sets

In [ ]:
lib.predictors.cache_clear()
rows = []
for s in lib.SETS:
    for r in lib.RUNGS:
        num, cat = lib.predictors(s, r)
        rows.append({"set": lib.SET_LABEL[s], "rung": r, "predictors": len(num) + len(cat),
                     "which": ", ".join(list(num) + list(cat)) if len(num) + len(cat) <= 10 else "…"})
        repro.check(f"{s}/{r}: the predictors are the study's",
                    sorted(list(num) + list(cat)) == sorted(study["sets"][s][r]))
print(pd.DataFrame(rows).to_string(index=False))
repro.report()